# Solución 9: Búsqueda de raíces — Pelota flotante (inodoro)

Origen: `01. Fundamental Algorithms/04.RootSearching/Roots_searching.pdf`

---

## Problema físico: pelota flotante (flotador de inodoro)

Una pelota esférica flota en agua. El equilibrio de fuerzas (peso = empuje) conduce a la ecuación:

$$f(x) = x^3 - 0.165x^2 + 3.993\times10^{-4} = 0$$

donde $x$ es la profundidad a la que se sumerge la pelota (en metros).

Esta ecuación cúbica se resolverá usando tres métodos: **bisección**, **Newton-Raphson** y **secante**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# FUNCIÓN Y DERIVADA
# ============================================================
def f(x):
    """Ecuación de la pelota flotante."""
    return x**3 - 0.165*x**2 + 3.993e-4

def df(x):
    """Derivada de f."""
    return 3*x**2 - 2*0.165*x

# Visualización de la función
x_plot = np.linspace(-0.05, 0.20, 400)
plt.figure(figsize=(9, 4))
plt.plot(x_plot, f(x_plot), 'royalblue', linewidth=2.5, label=r'$f(x)=x^3-0.165x^2+3.993\times10^{-4}$')
plt.axhline(0, color='black', linewidth=1)
plt.axvline(0, color='black', linewidth=0.5)
plt.xlabel('$x$ (profundidad de inmersión [m])')
plt.ylabel('$f(x)$')
plt.title('Función de la pelota flotante')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("f(0)    =", f(0))
print("f(0.05) =", f(0.05))
print("f(0.10) =", f(0.10))
print("f(0.15) =", f(0.15))
print("→ La raíz está entre x=0 y x=0.05 (cambio de signo)")

---
## Parte 1: Método de Bisección

El método de bisección divide sucesivamente el intervalo $[x_l, x_u]$ a la mitad y selecciona el subintervalo donde hay cambio de signo.

In [ ]:
# ============================================================
# MÉTODO DE BISECCIÓN
# ============================================================
def biseccion(f, xl, xu, tol=1e-6, max_iter=100):
    """
    Método de bisección.
    Devuelve (raíz, iteraciones, tabla).
    """
    if f(xl) * f(xu) > 0:
        raise ValueError("No hay cambio de signo en el intervalo dado")
    
    tabla = []
    xr_ant = xl
    
    for i in range(1, max_iter + 1):
        xr = (xl + xu) / 2
        fxr = f(xr)
        
        if i > 1:
            err_rel = abs((xr - xr_ant) / xr) * 100
        else:
            err_rel = np.nan
        
        tabla.append({'Iter': i, 'xl': xl, 'xu': xu, 'xr': xr,
                      'f(xr)': fxr, 'err_rel%': err_rel})
        
        if i > 1 and err_rel < tol * 100:
            break
        
        if f(xl) * fxr < 0:
            xu = xr
        else:
            xl = xr
        
        xr_ant = xr
    
    return xr, i, pd.DataFrame(tabla)


xr_bis, n_bis, tabla_bis = biseccion(f, 0.0, 0.05, tol=1e-6)

print("=" * 65)
print("MÉTODO DE BISECCIÓN — pelota flotante")
print("=" * 65)
pd.set_option('display.float_format', lambda x: f'{x:.8f}')
print(tabla_bis.to_string(index=False))
print("=" * 65)
print(f"Raíz hallada  : x = {xr_bis:.8f} m")
print(f"Iteraciones   : {n_bis}")
print(f"f(xr) = {f(xr_bis):.2e}")

---
## Parte 2: Método Newton-Raphson

$$x_{i+1} = x_i - \frac{f(x_i)}{f'(x_i)}$$

In [ ]:
# ============================================================
# MÉTODO NEWTON-RAPHSON
# ============================================================
def newton_raphson(f, df, x0, tol=1e-6, max_iter=50):
    """
    Método Newton-Raphson.
    Devuelve (raíz, iteraciones, tabla).
    """
    tabla = []
    xi = x0
    
    for i in range(1, max_iter + 1):
        fxi  = f(xi)
        dfxi = df(xi)
        
        if abs(dfxi) < 1e-15:
            raise ZeroDivisionError("Derivada cero en Newton-Raphson")
        
        xi1 = xi - fxi / dfxi
        err_rel = abs((xi1 - xi) / xi1) * 100 if abs(xi1) > 1e-15 else np.nan
        
        tabla.append({'Iter': i, 'xi': xi, 'f(xi)': fxi,
                      "f'(xi)": dfxi, 'xi+1': xi1, 'err_rel%': err_rel})
        xi = xi1
        
        if err_rel is not np.nan and err_rel < tol * 100:
            break
    
    return xi, i, pd.DataFrame(tabla)


xr_nr, n_nr, tabla_nr = newton_raphson(f, df, x0=0.05, tol=1e-6)

print("=" * 70)
print("MÉTODO NEWTON-RAPHSON — pelota flotante  (x0=0.05)")
print("=" * 70)
print(tabla_nr.to_string(index=False))
print("=" * 70)
print(f"Raíz hallada  : x = {xr_nr:.8f} m")
print(f"Iteraciones   : {n_nr}")
print(f"f(xr) = {f(xr_nr):.2e}")

---
## Parte 3: Método de la Secante

**Ejercicio para entregar:** Utiliza el método de la secante para hallar la profundidad $x$ a la que se sumerge la pelota. Realizar **3 iteraciones** y hallar el error relativo aproximado al final de cada una.

$$x_{i+1} = x_i - \frac{f(x_i)(x_i - x_{i-1})}{f(x_i) - f(x_{i-1})}$$

In [ ]:
# ============================================================
# MÉTODO DE LA SECANTE
# ============================================================
def secante(f, x0, x1, tol=1e-6, max_iter=50):
    """
    Método de la secante.
    Devuelve (raíz, iteraciones, tabla).
    """
    tabla = []
    xi_1 = x0   # x_{i-1}
    xi   = x1   # x_i
    
    for i in range(1, max_iter + 1):
        fxi_1 = f(xi_1)
        fxi   = f(xi)
        
        denom = fxi - fxi_1
        if abs(denom) < 1e-15:
            raise ZeroDivisionError("División por cero en el método secante")
        
        xi1 = xi - fxi * (xi - xi_1) / denom
        err_rel = abs((xi1 - xi) / xi1) * 100 if abs(xi1) > 1e-15 else np.nan
        
        # Dígitos significativos correctos
        if not np.isnan(err_rel) and err_rel > 0:
            dig_sig = int(np.floor(2 - np.log10(err_rel)))
        else:
            dig_sig = '—'
        
        tabla.append({'Iter': i, 'x_{i-1}': xi_1, 'x_i': xi,
                      'x_{i+1}': xi1, 'f(x_i)': fxi,
                      'err_rel%': err_rel, 'dig_sig': dig_sig})
        
        xi_1 = xi
        xi   = xi1
        
        if not np.isnan(err_rel) and err_rel < tol * 100:
            break
    
    return xi, i, pd.DataFrame(tabla)


# El ejercicio pide x0, x1 iniciales (valores razonables)
xr_sec, n_sec, tabla_sec = secante(f, x0=0.0, x1=0.05, tol=1e-6)

print("=" * 80)
print("MÉTODO DE LA SECANTE — pelota flotante  (x0=0.0, x1=0.05)")
print("=" * 80)
print(tabla_sec.to_string(index=False))
print("=" * 80)
print(f"Raíz hallada  : x = {xr_sec:.8f} m")
print(f"Iteraciones   : {n_sec}")
print(f"f(xr) = {f(xr_sec):.2e}")
print()
print("--- Primeras 3 iteraciones (ejercicio) ---")
print(tabla_sec.head(3).to_string(index=False))

In [ ]:
# ============================================================
# COMPARACIÓN FINAL
# ============================================================
print("\n" + "=" * 50)
print("COMPARACIÓN DE MÉTODOS")
print("=" * 50)
print(f"Bisección       : x = {xr_bis:.8f} m  ({n_bis} iter)")
print(f"Newton-Raphson  : x = {xr_nr:.8f} m  ({n_nr} iter)")
print(f"Secante         : x = {xr_sec:.8f} m  ({n_sec} iter)")
print("=" * 50)

# Visualización de convergencia
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, tabla, xr, nombre, color in zip(
        axes,
        [tabla_bis['xr'], tabla_nr['xi+1'], tabla_sec['x_{i+1}']],
        [xr_bis, xr_nr, xr_sec],
        ['Bisección', 'Newton-Raphson', 'Secante'],
        ['#e74c3c', '#27ae60', '#2980b9']):

    errores = np.abs(tabla.values - xr)
    ax.semilogy(range(1, len(errores)+1), errores + 1e-15, 'o-', color=color, linewidth=2)
    ax.set_title(nombre, fontsize=12)
    ax.set_xlabel('Iteración')
    ax.set_ylabel('|error|')
    ax.grid(True, alpha=0.3)

plt.suptitle('Convergencia de métodos — pelota flotante', fontsize=13)
plt.tight_layout()
plt.show()

---
## Parte 4: Newton-Raphson para ecuaciones simultáneas no lineales

**Ejercicio:** Solucione el sistema:
$$u(x,y) = xy - 2 = 0$$
$$v(x,y) = x^2 + y - 5 = 0$$
con valores iniciales $(x_0, y_0) = (3, 4)$. Iterar hasta que el error relativo sea menor al $0.05$.

La actualización en cada paso se obtiene resolviendo:
$$\mathbf{J}\begin{pmatrix}\Delta x \\ \Delta y\end{pmatrix} = -\begin{pmatrix}u \\ v\end{pmatrix}$$

donde $\mathbf{J}$ es el Jacobiano:
$$\mathbf{J} = \begin{pmatrix}\partial u/\partial x & \partial u/\partial y \\ \partial v/\partial x & \partial v/\partial y\end{pmatrix} = \begin{pmatrix}y & x \\ 2x & 1\end{pmatrix}$$

In [ ]:
# ============================================================
# NEWTON-RAPHSON PARA SISTEMA DE 2 ECUACIONES
# ============================================================
def u(x, y): return x*y - 2
def v(x, y): return x**2 + y - 5

def jacobiano(x, y):
    """Matriz Jacobiana del sistema."""
    return np.array([[y, x],
                     [2*x, 1]])


def nr_sistema(u, v, J, x0, y0, tol=0.05, max_iter=20):
    """
    Newton-Raphson para sistema de dos ecuaciones.
    Itera hasta que AMBOS errores relativos sean < tol.
    """
    xi, yi = x0, y0
    tabla = []
    
    for i in range(1, max_iter + 1):
        F  = np.array([-u(xi, yi), -v(xi, yi)])
        Ji = J(xi, yi)
        delta = np.linalg.solve(Ji, F)
        
        xi1 = xi + delta[0]
        yi1 = yi + delta[1]
        
        err_x = abs(delta[0] / xi1) if abs(xi1) > 1e-15 else np.nan
        err_y = abs(delta[1] / yi1) if abs(yi1) > 1e-15 else np.nan
        
        tabla.append({
            'Iter': i,
            'xi': xi, 'yi': yi,
            'xi+1': xi1, 'yi+1': yi1,
            'u(xi,yi)': u(xi, yi), 'v(xi,yi)': v(xi, yi),
            'err_x': err_x, 'err_y': err_y
        })
        
        xi, yi = xi1, yi1
        
        if err_x < tol and err_y < tol:
            break
    
    return xi, yi, i, pd.DataFrame(tabla)


xr_s, yr_s, n_s, tabla_s = nr_sistema(u, v, jacobiano, x0=3.0, y0=4.0, tol=0.05)

print("=" * 80)
print("NEWTON-RAPHSON — Sistema no lineal: xy=2, x²+y=5")
print("Valores iniciales: (x0, y0) = (3, 4)")
print("=" * 80)
print(tabla_s.to_string(index=False))
print("=" * 80)
print(f"Solución hallada : x = {xr_s:.6f},  y = {yr_s:.6f}")
print(f"Verificación     : u = {u(xr_s,yr_s):.2e},  v = {v(xr_s,yr_s):.2e}")
print(f"Iteraciones      : {n_s}")

In [ ]:
# Visualización del sistema y la raíz
xg = np.linspace(0.5, 4.0, 300)
yg = np.linspace(-1.0, 5.0, 300)
Xg, Yg = np.meshgrid(xg, yg)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(Xg, Yg, Xg*Yg - 2, levels=[0], colors='royalblue', linewidths=2)
ax.contour(Xg, Yg, Xg**2 + Yg - 5, levels=[0], colors='crimson', linewidths=2)

# Trayectoria iterativa
ax.plot(tabla_s['xi'].tolist() + [xr_s],
        tabla_s['yi'].tolist() + [yr_s],
        'ko--', markersize=6, linewidth=1.2, label='Iteraciones NR')
ax.plot(xr_s, yr_s, 'g*', markersize=16, zorder=5, label=f'Raíz ({xr_s:.3f}, {yr_s:.3f})')

ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title('Newton-Raphson sistema no lineal\n$u: xy=2$ (azul) — $v: x^2+y=5$ (rojo)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()